In [44]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, RobustScaler, OrdinalEncoder, LabelEncoder
)
from sklearn.metrics import mean_squared_error
from sklearn.svm import SVR

In [26]:
df = sns.load_dataset('tips')

In [27]:
y_df = df[['tip']]
X_df = df.drop(columns=['tip'], axis=1)
X_df

nominal_columns = ['sex', 'smoker', 'day', 'time']
continuos_columns = ['total_bill', 'size']

In [28]:
nominal_preprocessor = Pipeline(
    steps=[
        # Tratamento de dados faltantes
        ('missing', SimpleImputer(strategy='most_frequent')), 
        # Codificação de variáveis
        ('encoding', OneHotEncoder(sparse=False, handle_unknown='ignore')), 
        # Normalização de variáveis
        ('normalization', StandardScaler()) 
    ]
)
continuous_preprocessor = Pipeline(
    steps=[
        # Tratamento de dados faltantes
        ('missing', KNNImputer(n_neighbors=5)), 
        # Normalização
        ('normalization', RobustScaler())
    ]
)

preprocessor = ColumnTransformer([
    ('nominal', nominal_preprocessor, nominal_columns),
    ('continuos', continuous_preprocessor, continuos_columns)
])

In [29]:
preprocessor.fit(X_df)

ColumnTransformer(transformers=[('nominal',
                                 Pipeline(steps=[('missing',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoding',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse=False)),
                                                 ('normalization',
                                                  StandardScaler())]),
                                 ['sex', 'smoker', 'day', 'time']),
                                ('continuos',
                                 Pipeline(steps=[('missing', KNNImputer()),
                                                 ('normalization',
                                                  RobustScaler())]),
                                 ['total_bill', 'size'])])

In [30]:
preprocessor.transform(X_df).shape

(244, 12)

In [31]:
X = preprocessor.transform(X_df)
y = y_df.values

 - entradas: $\mathbf{X}$ 
 - saídas: $\boldsymbol{y}$
 - coeficientes: $\boldsymbol{\beta}$
 
$$
\mathbf{X} \boldsymbol{\beta} = \boldsymbol{y}
$$

Estimar o $\boldsymbol{\beta}$:

$$
\hat{\boldsymbol{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\boldsymbol{y}
$$


In [32]:
model = LinearRegression()
model.fit(X, y)
y_hat = model.predict(X)

In [43]:
mse = mean_squared_error(y, y_hat)
mse

1.0103574374656248

In [45]:
model = SVR()
model.fit(X, y)
y_hat = model.predict(X)

mse = mean_squared_error(y, y_hat)
mse

0.9135476585629827

In [40]:
foo = pd.DataFrame({
    'total_bill': [50],
    'sex': ['Female'],
    'smoker': ['No'],
    'time': ['Dinner'],
    'size': [2],
    'day': ['Sat']
})

In [46]:
model.predict(preprocessor.transform(foo))

array([4.69166249])